## Лабораторная работа 10: Сигналы и системы

### Упражнение 10.1: Линейная и циклическая свертка

In [ ]:
import sys
sys.path.insert(0, '../ThinkDSP/code')

from thinkdsp import read_wave
from thinkdsp import decorate
import matplotlib.pyplot as plt
import numpy as np
import os

### Проблема циклической свертки

При умножении DFT сигнала на передаточную функцию получается циклическая свертка, которая предполагает периодичность сигнала. Это может привести к артефактам в начале выходного сигнала.

### Загрузка импульсной характеристики

In [ ]:
# Загрузка звука выстрела как импульсной характеристики
response = read_wave('../ThinkDSP/code/180960__kleeb__gunshot.wav')

# Обрезка начала
start = 0.12
response = response.segment(start=start)
response.shift(-start)

# Обрезка до 2^16 элементов
response.truncate(2**16)

response.normalize()
response.plot()
plt.title('Импульсная характеристика')
decorate(xlabel='Время (с)', ylabel='Амплитуда')
plt.show()

### Спектр импульсной характеристики

In [ ]:
transfer = response.make_spectrum()
transfer.plot()
plt.title('Передаточная функция')
decorate(xlabel='Частота (Гц)', ylabel='Амплитуда')
plt.show()

### Загрузка входного сигнала

In [ ]:
# Загрузка музыкального фрагмента
wave = read_wave('../ThinkDSP/code/92002__jcveliz__violin-origional.wav')

# Берем начало
wave = wave.segment(duration=2.3)
wave.truncate(2**16)

wave.normalize()
wave.plot()
plt.title('Входной сигнал (скрипка)')
decorate(xlabel='Время (с)', ylabel='Амплитуда')
plt.show()

# Прослушивание
wave.make_audio()

### Свертка без zero-padding (с артефактами)

In [ ]:
# Свертка без дополнения нулями
spectrum = wave.make_spectrum()
output_spectrum = spectrum * transfer
output_wave = output_spectrum.make_wave()
output_wave.normalize()

# Визуализация начала сигнала
segment = output_wave.segment(duration=0.5)
segment.plot()
plt.title('Выход без zero-padding (видны артефакты в начале)')
decorate(xlabel='Время (с)', ylabel='Амплитуда')
plt.show()

# Прослушивание
output_wave.make_audio()

**Комментарий:** В начале сигнала видны артефакты из-за циклической свертки - конец сигнала "заворачивается" в начало.

### Свертка с zero-padding (без артефактов)

In [ ]:
# Дополнение нулями до 2^17 элементов
wave_padded = wave.copy()
wave_padded.zero_pad(2**17)

response_padded = response.copy()
response_padded.zero_pad(2**17)

# Свертка с дополнением нулями
spectrum_padded = wave_padded.make_spectrum()
transfer_padded = response_padded.make_spectrum()
output_spectrum_padded = spectrum_padded * transfer_padded
output_wave_padded = output_spectrum_padded.make_wave()
output_wave_padded.normalize()

# Визуализация начала сигнала
segment_padded = output_wave_padded.segment(duration=0.5)
segment_padded.plot()
plt.title('Выход с zero-padding (артефакты устранены)')
decorate(xlabel='Время (с)', ylabel='Амплитуда')
plt.show()

# Прослушивание
output_wave_padded.make_audio()

**Комментарий:** Дополнение нулями устраняет эффект "заворачивания" и дает корректную линейную свертку.

### Сравнение результатов

In [ ]:
# Сравнение начала сигналов
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Без zero-padding
segment1 = output_wave.segment(duration=0.5)
axes[0].plot(segment1.ts, segment1.ys)
axes[0].set_title('Без zero-padding (с артефактами)')
axes[0].set_xlabel('Время (с)')
axes[0].set_ylabel('Амплитуда')
axes[0].grid(True, alpha=0.3)

# С zero-padding
segment2 = output_wave_padded.segment(duration=0.5)
axes[1].plot(segment2.ts, segment2.ys)
axes[1].set_title('С zero-padding (без артефактов)')
axes[1].set_xlabel('Время (с)')
axes[1].set_ylabel('Амплитуда')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Комментарий:** 
- Циклическая свертка предполагает периодичность сигнала
- Без zero-padding конец сигнала "заворачивается" в начало
- Дополнение нулями до удвоенной длины устраняет эффект заворачивания
- Использование степеней двойки (2^16, 2^17) оптимизирует работу FFT